In [3]:
import urllib.parse
import warnings
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SAWarning

# SAWarning 무시 (버전 인식 경고 제거)
warnings.filterwarnings("ignore", category=SAWarning)

# DB 연결 정보
server = "127.0.0.1"
database = "kamtec"
username = "sa"
password = "1234"
driver = "ODBC Driver 18 for SQL Server"

# 연결 문자열 생성
params = urllib.parse.quote_plus(
    f"DRIVER={driver};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    f"Encrypt=no;"
    f"TrustServerCertificate=yes;"
)

# 엔진 생성 (성능 + 안정성 옵션 추가)
engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True,
    pool_pre_ping=True,
    pool_recycle=3600
)

# 데이터 로드 함수 (Pandas DataFrame으로 바로 변환)
def fetch_data(query):
    with engine.connect() as conn:
        df = pd.read_sql(text(query), conn)
    return df

In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

# 1. 특성 공학 단계
def apply_feature_engineering(df):
    # 시간 순 정렬 필수
    df = df.sort_values('s001').reset_index(drop=True) 
    
    # 기본 전류 (Current)
    # 이동 평균 (0.5초 간격의 추세)
    df['ma_12'] = df['F10_01'].rolling(window=12).mean()
    # 이동 표준편차 (전류의 떨림/노이즈)
    df['std_10'] = df['F10_01'].rolling(window=10).std()
    # 전류 변화율 (가속/감속 징후)
    df['delta'] = df['F10_01'].diff()
    # RMS (실효 에너지)
    df['rms_12'] = np.sqrt(df['F10_01'].pow(2).rolling(window=12).mean())
    
    # 결측치(NaN) 제거 및 필요한 컬럼만 선택
    features = ['F10_01', 'ma_12', 'std_10', 'delta', 'rms_12']
    df = df.dropna().reset_index(drop=True)
    
    return df[features]

# 데이터 가져오기 (정상 동작 데이터라고 가정)
query = """
SELECT s001, F10_01 
FROM lmsCurrent_Copy WITH (NOLOCK) 
WHERE F10_01 IS NOT NULL AND F10_01 > 0 
ORDER BY id ASC
"""

raw_df = fetch_data(query)
feature_df = apply_feature_engineering(raw_df)

print('feature_df: ', feature_df)

feature_df:            F10_01     ma_12    std_10  delta    rms_12
0          0.286  0.245750  0.036366  0.000  0.248093
1          0.286  0.251500  0.035631  0.000  0.253855
2          0.286  0.257250  0.033330  0.000  0.259489
3          0.200  0.255833  0.036363 -0.086  0.258349
4          0.200  0.254417  0.039080  0.000  0.257203
...          ...       ...       ...    ...       ...
51511400   0.031  0.038750  0.009102 -0.006  0.039788
51511401   0.031  0.037167  0.008496  0.000  0.038142
51511402   0.040  0.036583  0.008086  0.009  0.037471
51511403   0.040  0.036000  0.007406  0.000  0.036788
51511404   0.040  0.037083  0.006143  0.000  0.037761

[51511405 rows x 5 columns]


In [10]:
# 2. 정규화 및 시퀀스 생성
scaler = StandardScaler()
scaled_data = scaler.fit_transform(feature_df).astype('float32')
joblib.dump(scaler, 'lms_scaler.pkl') # 나중에 쓰기 위해 저장

def create_sequences(data, window_size=20):
    sequences = []
    for i in range(len(data) - window_size + 1):
        sequences.append(data[i : i + window_size])
    return np.array(sequences)

# window_size=20 (100ms * 20 = 2초 패턴 학습)
X_train = create_sequences(scaled_data, window_size=20)
print(f"학습 데이터 형태: {X_train.shape}") # (Samples, 20, 5)

학습 데이터 형태: (51511386, 20, 5)


In [11]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import timeseries_dataset_from_array

# 1. 모델 빌드 (Warning 해결 버전)
def build_autoencoder(window_size, num_features):
    model = Sequential([
        # 명시적인 Input 레이어 추가로 Warning 제거
        Input(shape=(window_size, num_features)),
        # Encoder
        LSTM(32, activation='tanh', return_sequences=False),
        RepeatVector(window_size),
        # Decoder
        LSTM(32, activation='tanh', return_sequences=True),
        TimeDistributed(Dense(num_features))
    ])
    model.compile(optimizer='adam', loss='mae')
    return model

# 2. 데이터 제너레이터 생성 (MemoryError 해결 핵심)
# scaled_data: (전체 샘플 수, 5) 형태의 numpy array
# X_train을 따로 만들지 않고 바로 이 generator를 fit에 넣습니다.
batch_size = 128 # 메모리 상황에 따라 64, 128, 256 조정
window_size = 20

# 시퀀스를 미리 생성하지 않고, 학습 시점에 index만 참조하여 배치 생성
train_gen = timeseries_dataset_from_array(
    data=scaled_data,
    targets=scaled_data[window_size-1:], # Autoencoder이므로 대상을 자기 자신(시퀀스 끝점)으로 설정하거나, 
                                        # Generator 특성에 맞춰 조정
    sequence_length=window_size,
    batch_size=batch_size,
    shuffle=True # 학습용이므로 섞어주는 것이 좋습니다.
)
# 주의: Autoencoder는 입력과 출력이 같아야 하므로, 
# generator의 출력을 (X, X) 형태로 매핑해주는 작업이 필요할 수 있습니다.
train_gen = train_gen.map(lambda x, y: (x, x)) 

# 3. 모델 생성 및 학습
model = build_autoencoder(window_size, scaled_data.shape[1])
early_stop = EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)

# X_train 대신 train_gen을 직접 대입
history = model.fit(
    train_gen,
    epochs=50,
    callbacks=[early_stop],
    verbose=1
)

model.save('lms_autoencoder.keras')

Epoch 1/50
236456/402433 ━━━━━━━━━━━━━━━━━━━━ 21:50 8ms/step - loss: 0.0898

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

# 1. Generator를 사용하여 배치 단위로 예측 및 오차 계산
print("임계값 설정을 위한 오차 계산 중...")
train_mae_loss = []

# 학습 데이터 전체를 다 쓰기에 너무 많다면 
# train_gen.take(샘플수)를 사용하여 일부만 추출할 수도 있습니다.
for batch_x, _ in train_gen:
    batch_pred = model.predict(batch_x, verbose=0)
    # 각 샘플(배치 내)에 대한 MAE 계산
    mae = np.mean(np.abs(batch_pred - batch_x), axis=(1, 2))
    train_mae_loss.extend(mae)

train_mae_loss = np.array(train_mae_loss)

# 2. 임계값 설정 (99퍼센타일)
threshold = np.percentile(train_mae_loss, 99)
print(f"이상 탐지 임계값: {threshold}")

# 3. 시각화 (데이터가 너무 많으면 히스토그램이 느려지므로 10만 개 정도 샘플링)
plt.figure(figsize=(10, 6))
if len(train_mae_loss) > 100000:
    sample_loss = np.random.choice(train_mae_loss, 100000)
    plt.hist(sample_loss, bins=50, alpha=0.7)
    plt.title("Train MAE Loss Distribution (Sampled 100k)")
else:
    plt.hist(train_mae_loss, bins=50, alpha=0.7)
    plt.title("Train MAE Loss Distribution")

plt.axvline(threshold, color='r', linestyle='--', label=f'Threshold ({threshold:.4f})')
plt.xlabel("Reconstruction MAE loss")
plt.ylabel("Number of samples")
plt.legend()
plt.show()

In [ ]:
def predict_anomaly(new_raw_data):
    # 1. 특성 공학 및 정규화
    new_features = apply_feature_engineering(new_raw_data)
    new_scaled = scaler.transform(new_features)
    
    # 2. 시퀀스 생성
    new_seq = create_sequences(new_scaled, window_size=20)
    
    # 3. 복원 및 오차 계산
    new_pred = model.predict(new_seq)
    new_mae_loss = np.mean(np.abs(new_pred - new_seq), axis=(1, 2))
    
    # 4. 임계값과 비교
    is_anomaly = new_mae_loss > threshold
    return is_anomaly, new_mae_loss

# 실제 적용 시나리오
# query_test = "SELECT Timestamp, current FROM LCMR_Logs WHERE Timestamp > ..."
# test_df = fetch_data(query_test)
# anomalies, loss = predict_anomaly(test_df)